## Data pulling and prep

In [1]:
#Pull generated summaries after tuning
%store -r generated_text_list_2_1
%store -r generated_text_list_2_2
%store -r generated_text_list_3_1
%store -r generated_text_list_3_2

In [2]:
#looking at data
print(generated_text_list_2_1)

['Francisco and Barnardo, both sentinels, converge on Horatio and Marcellus. They reassure them that Horatio’s', 'Polonius, the Queen’s mother, tries to convince Polonius that he is innocent. Polonius tries to convince Polonius that Polo', 'Antony and Nell arrive at the Maskers, but Antony and Capulet wait for music to be played. They meet and greet the', 'Friar Lawrence and Romeo express their gratitude for their love. They express their gratitude to Romeo, despite the fact that he is a', 'Roderigo and Iago seek revenge for the slain consuls. They seek revenge, but Roderigo refuses', 'Desdemona and Emilia are reunited. Emilia and Desdemona discuss their relationship with Emilia. Emili', 'Nature bankrupts himself, revealing his wealth, and revealing his wealth. Nature reveals his wealth, revealing his wealth, and revealing', 'Muse teaches thee how to make him appear, long hence, as he shows now. Muse teaches thee how to make him']


In [4]:
#Pull human summaries
%store -r paired_summaries

In [5]:
#Looking at data
hamun_val_8_summaries=list()

hamun_val_8_summaries.append(paired_summaries["Hamlet-Act I-Scene I"][1])
hamun_val_8_summaries.append(paired_summaries["Hamlet-Act III-Scene IV"][1])

hamun_val_8_summaries.append(paired_summaries["Romeo_and_Juliet-Act I-Scene V"][1])
hamun_val_8_summaries.append(paired_summaries["Romeo_and_Juliet-Act II-Scene VI"][1])

hamun_val_8_summaries.append(paired_summaries["Othello-Act I-Scene I"][1])
hamun_val_8_summaries.append(paired_summaries["Othello-Act IV-Scene II"][1])

hamun_val_8_summaries.append(paired_summaries["Sonnet 67"][1])
hamun_val_8_summaries.append(paired_summaries["Sonnet 101"][1])

print(len(hamun_val_8_summaries))

8


## Pulling scoring metrics

In [6]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

from bert_score import score as bertscore

import torch

from transformers import AutoModel


/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [7]:
#rouge and cosine similarity methods (created with the help of ChatGPT)
def compute_rouge(text1, text2):
    """
    Compute ROUGE between two texts.
    Returns F1 scores for ROUGE-1, ROUGE-2, ROUGE-L.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    # Note: scorer.score(reference, prediction)
    scores = scorer.score(text2, text1)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_cosine_similarity(text1, text2):
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    This avoids loading any transformer model and is stable in most environments.
    """
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])  # shape (2, vocab_size)

    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])  # (1,1)
    return float(cos_sim[0, 0])

def compute_bertscore(text1: str, text2: str):
    """
    Compute BERTScore (Precision, Recall, F1) between two texts.
    text1 = candidate
    text2 = reference
    """
    P, R, F1 = bertscore(
        cands=[text1],
        refs=[text2],
        lang="en"
    )
    return {
        "precision": float(P[0]),
        "recall": float(R[0]),
        "f1": float(F1[0]),
    }


## Looking at more examples

In [ ]:
#defining test text pieces

#different meaning | little overlap
text1a = "Scientists recently discovered a new species of fish living deep in the Pacific Ocean. The creature glows faintly in the dark."
text1b = "A city council voted to increase public transportation funding. New bus routes will be added later this year."

#different meaning | much overlap
text2a = "The company announced strong quarterly earnings. Investors were pleased with the revenue growth and rising profits."
text2b = "The company announced disappointing quarterly earnings. Investors were upset with the declining revenue and shrinking profits."

#similar meaning | little overlap
text3a = "The project wrapped up faster than we expected. Everyone contributed their expertise, and the final result exceeded the client's expectations."
text3b = "We finished the assignment ahead of schedule. The team’s combined skills produced an outcome that impressed the customer."

#similar meaning | much overlap
text4a = "The weather today is warm and sunny. Many people are going outside to enjoy the beautiful day."
text4b = "The weather today is sunny and warm. A lot of people are heading outside to enjoy the beautiful day."


In [10]:
#Finding scores of text results
print(compute_rouge(text1a, text1b))
print(compute_cosine_similarity(text1a, text1b))
print(compute_bertscore(text1a, text1b))




{'rouge1': 0.25, 'rouge2': 0.0, 'rougeL': 0.2}
0.2526482442031814


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.920782208442688, 'recall': 0.9210140705108643, 'f1': 0.9208981394767761}


In [11]:
print(compute_rouge(text2a, text2b))
print(compute_cosine_similarity(text2a, text2b))
print(compute_bertscore(text2a, text2b))

{'rouge1': 0.8333333333333333, 'rouge2': 0.5294117647058824, 'rougeL': 0.7222222222222222}
0.7756859246591444


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.9905509948730469, 'recall': 0.9868544936180115, 'f1': 0.9886992573738098}


In [12]:
print(compute_rouge(text3a, text3b))
print(compute_cosine_similarity(text3a, text3b))
print(compute_bertscore(text3a, text3b))

{'rouge1': 0.10256410256410256, 'rouge2': 0.0, 'rougeL': 0.10256410256410256}
0.023758558544135375


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.8609054088592529, 'recall': 0.8640084862709045, 'f1': 0.8624542355537415}


In [13]:
print(compute_rouge(text4a, text4b))
print(compute_cosine_similarity(text4a, text4b))
print(compute_bertscore(text4a, text4b))

{'rouge1': 0.75, 'rouge2': 0.4000000000000001, 'rougeL': 0.75}
0.6392306240536285


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'precision': 0.9696977138519287, 'recall': 0.9696977138519287, 'f1': 0.9696977138519287}


## Doing scoring and comparisons

In [35]:
rouge_8_model2_1_val_summary_eval = dict()
cosine_8_model2_1_val_summary_eval = dict()
bertscore_8_model2_1_val_summary_eval = dict()

rouge_8_model2_2_val_summary_eval = dict()
cosine_8_model2_2_val_summary_eval = dict()
bertscore_8_model2_2_val_summary_eval = dict()

rouge_8_model3_1_val_summary_eval = dict()
cosine_8_model3_1_val_summary_eval = dict()
bertscore_8_model3_1_val_summary_eval = dict()

rouge_8_model3_2_val_summary_eval = dict()
cosine_8_model3_2_val_summary_eval = dict()
bertscore_8_model3_2_val_summary_eval = dict()

scene_8_list = ["Hamlet-Act I-Scene I","Hamlet-Act III-Scene IV","Hamlet-Act III-Scene IV","Romeo_and_Juliet-Act II-Scene VI","Othello-Act I-Scene I","Othello-Act IV-Scene II","Sonnet 67","Sonnet 101"]

#comarison of 8 human summaries and 8 fine tuned summary
for i in range(8):
    
    #2_1
    rouge_scores2_1 = compute_rouge(hamun_val_8_summaries[i], generated_text_list_2_1[i])
    rouge_8_model2_1_val_summary_eval[scene_8_list[i]] = (rouge_scores2_1["rouge1"],rouge_scores2_1["rouge2"],rouge_scores2_1["rougeL"])
    
    cosine_scores2_1 = compute_cosine_similarity(hamun_val_8_summaries[i], generated_text_list_2_1[i])
    cosine_8_model2_1_val_summary_eval[scene_8_list[i]] = cosine_scores2_1
    
    cosine_scores2_1 = compute_bertscore(hamun_val_8_summaries[i], generated_text_list_2_1[i])
    bertscore_8_model2_1_val_summary_eval[scene_8_list[i]] = (cosine_scores2_1["precision"],cosine_scores2_1["recall"],cosine_scores2_1["f1"])

    #2_2
    rouge_scores2_2 = compute_rouge(hamun_val_8_summaries[i], generated_text_list_2_2[i])
    rouge_8_model2_2_val_summary_eval[scene_8_list[i]] = (rouge_scores2_2["rouge1"],rouge_scores2_2["rouge2"],rouge_scores2_2["rougeL"])
    
    cosine_scores2_2 = compute_cosine_similarity(hamun_val_8_summaries[i], generated_text_list_2_2[i])
    cosine_8_model2_2_val_summary_eval[scene_8_list[i]] = cosine_scores2_2
    
    cosine_scores2_2 = compute_bertscore(hamun_val_8_summaries[i], generated_text_list_2_2[i])
    bertscore_8_model2_2_val_summary_eval[scene_8_list[i]] = (cosine_scores2_2["precision"],cosine_scores2_2["recall"],cosine_scores2_2["f1"])

    #3_1
    rouge_scores3_1 = compute_rouge(hamun_val_8_summaries[i], generated_text_list_3_1[i])
    rouge_8_model3_1_val_summary_eval[scene_8_list[i]] = (rouge_scores3_1["rouge1"],rouge_scores3_1["rouge2"],rouge_scores3_1["rougeL"])
    
    cosine_scores3_1 = compute_cosine_similarity(hamun_val_8_summaries[i], generated_text_list_3_1[i])
    cosine_8_model3_1_val_summary_eval[scene_8_list[i]] = cosine_scores3_1
    
    cosine_scores3_1 = compute_bertscore(hamun_val_8_summaries[i], generated_text_list_3_1[i])
    bertscore_8_model3_1_val_summary_eval[scene_8_list[i]] = (cosine_scores3_1["precision"],cosine_scores3_1["recall"],cosine_scores3_1["f1"])
    
    #3_2
    rouge_scores3_2 = compute_rouge(hamun_val_8_summaries[i], generated_text_list_3_2[i])
    rouge_8_model3_2_val_summary_eval[scene_8_list[i]] = (rouge_scores3_2["rouge1"],rouge_scores3_2["rouge2"],rouge_scores3_2["rougeL"])
    
    cosine_scores3_2 = compute_cosine_similarity(hamun_val_8_summaries[i], generated_text_list_3_2[i])
    cosine_8_model3_2_val_summary_eval[scene_8_list[i]] = cosine_scores3_2
    
    cosine_scores3_2 = compute_bertscore(hamun_val_8_summaries[i], generated_text_list_3_2[i])
    bertscore_8_model3_2_val_summary_eval[scene_8_list[i]] = (cosine_scores3_2["precision"],cosine_scores3_2["recall"],cosine_scores3_2["f1"])
    


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

In [36]:
#Looking at data
print(rouge_8_model2_1_val_summary_eval)
print(cosine_8_model2_1_val_summary_eval)
print(bertscore_8_model2_1_val_summary_eval)

{'Hamlet-Act I-Scene I': (0.35000000000000003, 0.052631578947368425, 0.2), 'Hamlet-Act III-Scene IV': (0.16049382716049385, 0.012500000000000002, 0.1111111111111111), 'Romeo_and_Juliet-Act II-Scene VI': (0.2909090909090909, 0.0, 0.18181818181818182), 'Othello-Act I-Scene I': (0.13333333333333333, 0.0, 0.1111111111111111), 'Othello-Act IV-Scene II': (0.14035087719298248, 0.03571428571428572, 0.10526315789473684), 'Sonnet 67': (0.057971014492753624, 0.0, 0.057971014492753624), 'Sonnet 101': (0.03846153846153846, 0.0, 0.03846153846153846)}
{'Hamlet-Act I-Scene I': 0.25763122791834947, 'Hamlet-Act III-Scene IV': 0.36943631424733464, 'Romeo_and_Juliet-Act II-Scene VI': 0.2283825213939515, 'Othello-Act I-Scene I': 0.13583275647898282, 'Othello-Act IV-Scene II': 0.2696569582064946, 'Sonnet 67': 0.05142721437776338, 'Sonnet 101': 0.022108556648454115}
{'Hamlet-Act I-Scene I': (0.8584386110305786, 0.8776766061782837, 0.8679510354995728), 'Hamlet-Act III-Scene IV': (0.8217113018035889, 0.8666166

In [16]:
#Pull pre-validation scores
%store -r rouge_val_summary_eval
%store -r cosine_val_summary_eval
%store -r bertscore_val_summary_eval

In [ ]:
#Making dicts which is rogue scores pre-training for the 8 were looking at
desired_keys=["Hamlet-Act I-Scene I","Hamlet-Act III-Scene IV","Hamlet-Act III-Scene IV","Romeo_and_Juliet-Act II-Scene VI","Othello-Act I-Scene I","Othello-Act IV-Scene II","Sonnet 67","Sonnet 101"]

rouge_8_pretrain_val_summary_eval = {key: rouge_val_summary_eval[key] for key in desired_keys if key in rouge_val_summary_eval}
cosine_8_pretrain_val_summary_eval = {key: cosine_val_summary_eval[key] for key in desired_keys if key in cosine_val_summary_eval}
bertscore_8_pretrain_val_summary_eval = {key: bertscore_val_summary_eval[key] for key in desired_keys if key in bertscore_val_summary_eval}


In [38]:
#Looking at data
print(rouge_8_pretrain_val_summary_eval)
print(cosine_8_pretrain_val_summary_eval)
print(bertscore_8_pretrain_val_summary_eval)

{'Hamlet-Act I-Scene I': (0.1111111111111111, 0.0, 0.05555555555555555), 'Hamlet-Act III-Scene IV': (0.09302325581395349, 0.0, 0.09302325581395349), 'Romeo_and_Juliet-Act II-Scene VI': (0.08791208791208792, 0.0, 0.06593406593406594), 'Othello-Act I-Scene I': (0.03773584905660377, 0.0, 0.03773584905660377), 'Othello-Act IV-Scene II': (0.1702127659574468, 0.014388489208633094, 0.1276595744680851), 'Sonnet 67': (0.25641025641025644, 0.0, 0.20512820512820512), 'Sonnet 101': (0.16129032258064516, 0.06666666666666667, 0.12903225806451613)}
{'Hamlet-Act I-Scene I': 0.051823247345752686, 'Hamlet-Act III-Scene IV': 0.40552405212948167, 'Romeo_and_Juliet-Act II-Scene VI': 0.06005846140631499, 'Othello-Act I-Scene I': 0.005675957323562465, 'Othello-Act IV-Scene II': 0.1666497989472491, 'Sonnet 67': 0.1632575553918144, 'Sonnet 101': 0.07242968902588375}
{'Hamlet-Act I-Scene I': (0.7753083109855652, 0.8183489441871643, 0.7962474226951599), 'Hamlet-Act III-Scene IV': (0.7916324138641357, 0.810408771

In [39]:
#Store all variables with info on these 8 (comparisons will happen in next notebook)
%store rouge_8_pretrain_val_summary_eval
%store cosine_8_pretrain_val_summary_eval
%store bertscore_8_pretrain_val_summary_eval

%store rouge_8_model2_1_val_summary_eval
%store cosine_8_model2_1_val_summary_eval
%store bertscore_8_model2_1_val_summary_eval

%store rouge_8_model2_2_val_summary_eval
%store cosine_8_model2_2_val_summary_eval
%store bertscore_8_model2_2_val_summary_eval

%store rouge_8_model3_1_val_summary_eval
%store cosine_8_model3_1_val_summary_eval
%store bertscore_8_model3_1_val_summary_eval

%store rouge_8_model3_2_val_summary_eval
%store cosine_8_model3_2_val_summary_eval
%store bertscore_8_model3_2_val_summary_eval





Stored 'rouge_8_pretrain_val_summary_eval' (dict)
Stored 'cosine_8_pretrain_val_summary_eval' (dict)
Stored 'bertscore_8_pretrain_val_summary_eval' (dict)
Stored 'rouge_8_model2_1_val_summary_eval' (dict)
Stored 'cosine_8_model2_1_val_summary_eval' (dict)
Stored 'bertscore_8_model2_1_val_summary_eval' (dict)
Stored 'rouge_8_model2_2_val_summary_eval' (dict)
Stored 'cosine_8_model2_2_val_summary_eval' (dict)
Stored 'bertscore_8_model2_2_val_summary_eval' (dict)
Stored 'rouge_8_model3_1_val_summary_eval' (dict)
Stored 'cosine_8_model3_1_val_summary_eval' (dict)
Stored 'bertscore_8_model3_1_val_summary_eval' (dict)
Stored 'rouge_8_model3_2_val_summary_eval' (dict)
Stored 'cosine_8_model3_2_val_summary_eval' (dict)
Stored 'bertscore_8_model3_2_val_summary_eval' (dict)
